# NER bằng model đã fine-tune trên 2.txt

Notebook này dùng `AutoModelForTokenClassification`, tức là NER thật, không dùng `CANDIDATE_TERMS` hay rule thủ công nữa.

Lưu ý: hiện chưa thấy checkpoint ViHealthBERT đã fine-tune NER public. Vì vậy demo này dùng checkpoint public `leduckhai/VietMed-NER/xlm-roberta-base-VietMed-NER`. Nếu sau này có checkpoint ViHealthBERT NER, chỉ cần đổi `model_repo` / `model_subfolder`.

In [2]:
import torch
from transformers import AutoModelForTokenClassification, AutoTokenizer

In [3]:
MODEL_NAME = "demdecuong/vihealthbert-base-word"

LABELS = [
    "O",
    "B-bien_phap_chan_doan",
    "I-bien_phap_chan_doan",
    "B-bien_phap_dieu_tri",
    "I-bien_phap_dieu_tri",
    "B-nguyen_nhan_benh",
    "I-nguyen_nhan_benh",
    "B-ten_benh",
    "I-ten_benh",
    "B-trieu_chung_benh",
    "I-trieu_chung_benh",
]

label2id = {label: index for index, label in enumerate(LABELS)}
id2label = {index: label for label, index in label2id.items()}

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    use_fast=False
)

model = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(LABELS),
    label2id=label2id,
    id2label=id2label,
)

text = "Bệnh nhân bị đau đầu và sốt cao."

inputs = tokenizer(
    text,
    return_tensors="pt",
    truncation=True,
    max_length=128
)

model.eval()

with torch.no_grad():
    outputs = model(**inputs)

predicted_ids = outputs.logits.argmax(dim=-1)[0]
tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])

for token, predicted_id in zip(tokens, predicted_ids):
    label = id2label[predicted_id.item()]
    print(f"{token:20} {label}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/817 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/895k [00:00<?, ?B/s]

bpe.codes:   0%|          | 0.00/1.14M [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/540M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForTokenClassification LOAD REPORT from: demdecuong/vihealthbert-base-word
Key                 | Status     | 
--------------------+------------+-
pooler.dense.bias   | UNEXPECTED | 
pooler.dense.weight | UNEXPECTED | 
classifier.bias     | MISSING    | 
classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


<s>                  B-trieu_chung_benh
Bệnh                 B-nguyen_nhan_benh
nhân                 I-trieu_chung_benh
bị                   I-ten_benh
đau                  B-trieu_chung_benh
đầu                  B-trieu_chung_benh
và                   B-trieu_chung_benh
sốt                  I-ten_benh
cao@@                I-ten_benh
.                    I-ten_benh
</s>                 B-trieu_chung_benh


model.safetensors:   0%|          | 0.00/540M [00:00<?, ?B/s]